# Bahia (BA) — Complexidade de Gestão e Esforço Docente (2025)

## Pergunta de Pesquisa

> **Nas escolas da Bahia (BA) em 2025, escolas com maior nível de complexidade de gestão apresentam maior proporção de docentes em altos níveis de esforço (níveis 5 e 6)?**

**Hipótese:** Escolas mais complexas de gerir tendem a sobrecarregar mais seus professores, resultando em percentuais maiores de docentes nos níveis críticos de esforço (5 e 6).

**Fonte dos dados:** Censo Escolar 2025 — INEP  
**Base de dados:** DuckDB (`project_data.duckdb`) — schema `main_serving`

In [9]:
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px

DB_PATH = 'project_data.duckdb'

con = duckdb.connect(database=DB_PATH, read_only=True)

# Carrega a tabela serving, filtrando apenas escolas com dado de esforço docente disponível.
# tx_esforco_docente_critico é NULL quando a escola não possui registro no IED.
df = con.execute("""
    SELECT *
    FROM main_serving.fct_desigualdade_gestao_esforco
    WHERE tx_esforco_docente_critico IS NOT NULL
""").df()

con.close()

print(f"Total de escolas com dados de esforço docente: {len(df):,}")
print(f"Níveis de complexidade presentes: {sorted(df['nivel_complexidade'].dropna().unique())}")

Total de escolas com dados de esforço docente: 11,790
Níveis de complexidade presentes: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


## Gráfico 1 — Resposta à Pergunta de Pesquisa

Média percentual de docentes em alto esforço (níveis 5+6) agrupada por nível de complexidade de gestão (ICG).  
Um padrão crescente nessa curva **confirma a hipótese** de que escolas mais complexas sobrecarregam mais seus professores.

In [10]:
# Agrega por nível de complexidade: calcula a média do esforço docente e conta escolas
df_complexidade = (
    df.groupby('nivel_complexidade', observed=True)
    .agg(
        media_esforco=('tx_esforco_docente_critico', 'mean'),
        qtd_escolas=('id_escola', 'count')
    )
    .reset_index()
    .sort_values('nivel_complexidade')
)

# Rótulo exibido em cima de cada barra: percentual médio + quantidade de escolas
df_complexidade['rotulo'] = df_complexidade.apply(
    lambda r: f"{r['media_esforco']:.1f}%<br>({int(r['qtd_escolas'])} escolas)", axis=1
)

fig1 = px.bar(
    df_complexidade,
    x='nivel_complexidade',
    y='media_esforco',
    text='rotulo',
    title='Média de Docentes em Alto Esforço por Nível de Complexidade de Gestão — BA 2025',
    labels={
        'nivel_complexidade': 'Nível de Complexidade de Gestão (ICG — 1 a 6)',
        'media_esforco': '% Médio de Docentes em Alto Esforço (Níveis 5+6)'
    },
    template='plotly_white',
    color='media_esforco',
    color_continuous_scale='Oranges'
)

fig1.update_traces(textposition='outside', textfont_size=11)
fig1.update_layout(
    title_font_size=17,
    showlegend=False,
    coloraxis_showscale=False,
    xaxis={'tickmode': 'linear', 'dtick': 1},
    yaxis={'range': [0, df_complexidade['media_esforco'].max() * 1.35]}
)

fig1.show()

## Análises Complementares

Os gráficos a seguir aprofundam a análise e fecham o raciocínio em cadeia:
- **Gráfico 2:** Quais redes concentram mais escolas de alta complexidade (níveis 5 e 6)?
- **Gráfico 3:** Quais redes têm maior média de docentes em alto esforço?

In [11]:
# Filtra Federal por ter apenas 1 escola — sem representatividade estatística
df_redes = df[df['no_dependencia'] != 'Federal']

# Agrega por rede e nível de complexidade, calculando proporção dentro de cada rede
df_rede_complex = (
    df_redes.groupby(['no_dependencia', 'nivel_complexidade'], observed=True)
    .size()
    .reset_index(name='qtd')
)
df_rede_complex['pct'] = (
    df_rede_complex['qtd']
    / df_rede_complex.groupby('no_dependencia')['qtd'].transform('sum')
)

fig2 = px.bar(
    df_rede_complex,
    x='no_dependencia',
    y='pct',
    color='nivel_complexidade',
    title='Distribuição dos Níveis de Complexidade por Rede Administrativa — BA 2025',
    labels={
        'no_dependencia': 'Rede Administrativa',
        'pct': 'Proporção de Escolas',
        'nivel_complexidade': 'Nível de Complexidade (ICG)'
    },
    template='plotly_white',
    color_continuous_scale='Oranges'
)

fig2.update_layout(
    title_font_size=17,
    yaxis_tickformat='.0%',
    coloraxis_colorbar_title='Nível ICG',
    xaxis={'categoryorder': 'total descending'}
)
fig2.update_traces(hovertemplate='<b>%{x}</b><br>Proporção: %{y:.1%}')

fig2.show()

In [12]:
# Filtra Federal por ter apenas 1 escola — sem representatividade estatística
# Agrega por rede administrativa: média de esforço docente e contagem de escolas
df_rede = (
    df[df['no_dependencia'] != 'Federal']
    .groupby('no_dependencia', observed=True)
    .agg(
        media_esforco=('tx_esforco_docente_critico', 'mean'),
        qtd_escolas=('id_escola', 'count')
    )
    .reset_index()
    .sort_values('media_esforco', ascending=False)
)

df_rede['rotulo'] = df_rede.apply(
    lambda r: f"{r['media_esforco']:.1f}%<br>({int(r['qtd_escolas'])} escolas)", axis=1
)

fig3 = px.bar(
    df_rede,
    x='no_dependencia',
    y='media_esforco',
    text='rotulo',
    title='Média de Docentes em Alto Esforço por Rede Administrativa — BA 2025',
    labels={
        'no_dependencia': 'Rede Administrativa',
        'media_esforco': '% Médio de Docentes em Alto Esforço (Níveis 5+6)'
    },
    template='plotly_white',
    color='media_esforco',
    color_continuous_scale='Blues'
)

fig3.update_traces(textposition='outside', textfont_size=11)
fig3.update_layout(
    title_font_size=17,
    showlegend=False,
    coloraxis_showscale=False,
    yaxis={'range': [0, df_rede['media_esforco'].max() * 1.35]}
)

fig3.show()

## Conclusão

Com base nos dados do Censo Escolar 2025 para a Bahia (11.790 escolas com dados de esforço docente disponíveis):

- A média de docentes em alto esforço (níveis 5+6) **cresce consistentemente** com o nível de complexidade de gestão (ICG): de **1,3% no nível 1** para **11,4% no nível 6**.
- **Confirmamos a hipótese:** escolas com maior complexidade de gestão tendem a ter maior proporção de docentes em situação crítica de esforço.
- O padrão se mantém mesmo ao segmentar por rede administrativa — escolas Estaduais concentram mais de 50% das suas unidades nos níveis 5 e 6, e registram a maior sobrecarga docente (16,8%).
- O nível 6 concentra apenas **244 escolas**, mas nelas a sobrecarga docente é significativamente mais alta — o que indica um problema concentrado, porém severo.